In [1]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain.text_splitter import CharacterTextSplitter
import os
from dotenv import load_dotenv

In [2]:
#Load API env
load_dotenv(dotenv_path=".env")
openai_api_key=os.getenv("OPENAI_API_KEY")

#Initialize LLM
llm= ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
#Load document and split
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([raw_text])

In [5]:
#Define prompts
map_prompt = PromptTemplate.from_template("Summarize:\n\n{context}")
reduce_prompt = PromptTemplate.from_template("Combine the following summaries into a single answer:\n\n{context}")


#Create map + reduce chains
map_chain = map_prompt | llm
reduce_chain = reduce_prompt | llm

#Use RunnableLambda to convert documents

def map_docs(docs):
    return [{"context": doc.page_content} for doc in docs]

def extract_texts(response):
    return {"context": "\n".join([res.content for res in response])}


#Combine using RunnableParallel (MapReduce style)
map_reduce_chain = (
    RunnableLambda(map_docs)
    | map_chain.map()
    | RunnableLambda(extract_texts)
    | reduce_chain
)

#Execute
result = map_reduce_chain.invoke(docs)
print("📄 Final Summary:\n", result.content)


📄 Final Summary:
 LangChain, developed by Harrison Chase, is a framework designed for building applications that leverage large language models (LLMs). It offers a range of features including retrieval-augmented generation (RAG), agents, memory, and tools, making it particularly useful for chatbots, document question-and-answer systems, and various AI workflows.
